# BSM L07G — Pochodzenie aplikacji, zaufanie w runtime i higiena kopii zapasowych (Android)

## Tryb pracy
To nie jest lab z Pythonem. Implementujesz rozwiązania w **Android Studio / Kotlin** w starterze projektu `lesson_g_app`.
Ten notebook służy jako:
- instrukcja krok-po-kroku (co otworzyć, gdzie kliknąć, czego szukać w kodzie),
- formularz odpowiedzi,
- mechanizm wysyłki odpowiedzi do backendu.

## Starter projektu
W tym repozytorium używasz folderu:
- `student/apps/lesson_g_app`

## Jak powstają odpowiedzi (ważne)
- **Zadanie 1 (G01)**: odpowiedź jest wysyłana automatycznie z aplikacji (nie ma w notebooku komórki „Wyślij”).
- **Zadania 2-4 (G02-G04)**: odpowiedzi wysyłasz z notebooka (komórki „Formularz odpowiedzi”).

## Literatura i dokumentacja (na ten lab)
Obowiązkowo odwołuj się do dokumentacji platformy (linki poniżej) i uzasadniaj decyzje bezpieczeństwa.

- Android Developers: App signing (overview)
- Android Developers: Data and file storage (SharedPreferences, internal storage)
- Android Developers: Auto Backup / Backup rules (w tym `android:allowBackup`, `android:fullBackupContent`, `android:dataExtractionRules`)
- Android Developers: `EncryptedSharedPreferences`, `MasterKey` (AndroidX Security Crypto)
- OWASP MASVS / MSTG: sekcje o przechowywaniu sekretów, uprawnieniach i prywatności

Z sylabusa: temat labu podpina się pod „Cechy bezpieczeństwa platformy Android” i praktykę „stosowania technik bezpieczeństwa zgodnie z dokumentacją deweloperską”.


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
# Komórka pomocnicza: formatowanie i wysyłanie odpowiedzi
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("
", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    # Wysyłka do backendu zadania
    wyslij_odpowiedz(task_id, final_answer)


In [ ]:
# Komórka pomocnicza: auto-wysyłka po wklejeniu odpowiedzi (Colab)
#
# W zadaniach G02-G04 nie klikamy osobnej komórki 'Wyślij'.
# Wklejasz wartość w textbox, a notebook sam wysyła odpowiedź.

_IMPORT_ERROR = None
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as _e:
    widgets = None
    _IMPORT_ERROR = _e

_SENT = {}

def autosend_text(task_id: str, title: str, placeholder: str = "", max_len: int = 220):
    if widgets is None:
        err = type(_IMPORT_ERROR).__name__ if _IMPORT_ERROR else "UnknownError"
        print("Brak ipywidgets. Jeśli używasz Google Colab, uruchom notebook w Colab. Błąd:", err)
        return

    header = widgets.HTML(f"<b>{title}</b>")
    box = widgets.Textarea(
        value="",
        placeholder=placeholder,
        description="",
        layout=widgets.Layout(width='100%', height='90px'),
    )
    out = widgets.Output()

    def _send(val: str):
        val = (val or "").strip()
        if not val:
            return
        if len(val) > max_len:
            with out:
                clear_output(wait=True)
                print(f"Odpowiedź ma {len(val)} znaków, a limit to {max_len}. Skróć i wklej ponownie.")
            return
        if _SENT.get(task_id) == val:
            return
        _SENT[task_id] = val
        with out:
            clear_output(wait=True)
            print(f"Wysyłam {task_id} ({len(val)} znaków)...")
            zapisz_i_wyslij(task_id, val)
            print("OK (wysłano).")

    def _on_change(change):
        if change.get('name') != 'value':
            return
        _send(change.get('new'))

    box.observe(_on_change)

    display(header)
    display(box)
    display(out)


# G01 — Manifest i audyt prywatności (odpowiedź wysyła aplikacja)

## Cel
Zobaczyć w praktyce, że deklaracja w `AndroidManifest.xml` i rzeczywiste użycie funkcji to dwie różne rzeczy.
W prawdziwych review (np. aplikacje sklepowe, audyty prywatności) patrzy się na:
- listę uprawnień,
- powód biznesowy i techniczny,
- moment prośby o uprawnienie (runtime vs manifest),
- minimalizację zakresu (principle of least privilege).

## Co masz zrobić
1. Otwórz projekt `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik manifestu: `app/src/main/AndroidManifest.xml`.
1. Zrób mapę: dla każdego `<uses-permission ...>` zapisz, która funkcja aplikacji go realnie potrzebuje:
- lokalizacja: mapka (pobranie bieżącej lokalizacji)
- kamera: zrobienie zdjęcia
- galeria: wybór zdjęcia
- internet: pobranie kafelka mapy z usługi zewnętrznej
1. Odpowiedz sobie na pytania kontrolne (nie wysyłasz ich do backendu, ale są kluczowe do zrozumienia):
- Czy wszystkie zadeklarowane uprawnienia są faktycznie potrzebne? Jeśli tak, to w jakiej sytuacji?
- Które z nich są „runtime permissions” i kiedy aplikacja powinna o nie prosić?
- Czy aplikacja ma sensowny fallback, jeśli użytkownik odmówi?

## Jak zaliczasz (automatycznie)
To zadanie jest powiązane z aplikacją.
1. Uruchom aplikację na emulatorze lub urządzeniu.
1. Wpisz swoje **Student ID** w ekranie aplikacji (sekcja „Student / Task 1”).
1. Kliknij „Request permissions” i przejdź cały przepływ.
1. Jeśli wszystko jest poprawnie, aplikacja sama wykona wysyłkę dla `G01`.

Uwaga: w tym notebooku **nie ma** komórki „Wyślij” dla G01.


# G02 — Bezpieczne przechowywanie klucza API (secure storage) + minimalizacja wycieków

## Kontekst teoretyczny
Klucz API do usługi zewnętrznej (np. map) to sekret. Najczęstsze przyczyny wycieku w aplikacjach mobilnych:
- hardcode w kodzie lub w zasobach (np. `strings.xml`, `BuildConfig`, pliki w `assets/`),
- logi (debug, crash reporting, analytics),
- kopie zapasowe i migracja (Auto Backup),
- niejawne udostępnienie przez komponenty (np. exportowany `ContentProvider`),
- przypadkowe zapisanie w publicznym storage.

W tym zadaniu budujesz minimalny, ale poprawny przepływ przechowywania sekretu tak, aby:
- był zapisany w miejscu przeznaczonym na dane wrażliwe,
- nie wypływał do logów ani do repo,
- aplikacja miała sensowne zachowanie, gdy sekretu nie ma.

## Co masz zrobić (krok po kroku w Android Studio)
1. Otwórz projekt `student/apps/lesson_g_app` w Android Studio.
1. Otwórz plik: `app/src/main/java/com/example/secretlab/MainActivity.kt`.
1. Znajdź stałą `MAP_API_KEY_PREF` i miejsce, gdzie klucz mapy jest pobierany do UI:
- w `LessonGStarterApp()` zobacz wywołanie `securePrefs.getString(MAP_API_KEY_PREF, null)` przekazywane do `ApiMapCard(...)`.
1. Otwórz implementację bezpiecznych preferencji:
- `app/src/main/java/com/example/secretlab/secure/SecurePrefs.kt`
- zwróć uwagę na `EncryptedSharedPreferences.create(...)` i `MasterKey.Builder(...).setKeyScheme(...)`.
1. Zaprojektuj brakujący krok użytkownika: skąd aplikacja ma wziąć klucz API.
   W starterze mapka pokazuje komunikat „Map API key missing.”, kiedy klucza nie ma.
   Twoim zadaniem jest uzupełnić przepływ tak, aby dało się bezpiecznie wprowadzić i zapisać klucz (bez ujawniania go w repo).

Wskazówki UX (bez gotowego kodu):
- najprostszy wariant to dodatkowe pole tekstowe w UI (Compose), które zapisuje wartość do bezpiecznych preferencji.
- pamiętaj o higienie: nie loguj klucza, nie pokazuj go w UI po zapisaniu, rozważ przycisk „Wyczyść klucz”.

## Na co uważać (bezpieczeństwo)
- Nie zapisuj sekretu w plain `SharedPreferences`.
- Nie wkładaj sekretu do `BuildConfig` ani do `strings.xml`.
- Nie loguj sekretu (nawet w debug).

## Linki do dokumentacji (czytaj i użyj w uzasadnieniu)
- EncryptedSharedPreferences:
  https://developer.android.com/reference/androidx/security/crypto/EncryptedSharedPreferences
- MasterKey:
  https://developer.android.com/reference/androidx/security/crypto/MasterKey
- Data and file storage (Android):
  https://developer.android.com/training/data-storage
- OWASP Mobile Application Security Verification Standard (MASVS):
  https://mas.owasp.org/

## Jak zdobyć kod zaliczeniowy (G02)
1. Uruchom testy unit:
- Android Studio: `View` -> `Tool Windows` -> `Gradle`, potem: `lesson_g_app` -> `app` -> `Tasks` -> `verification` -> `testDebugUnitTest`.
1. Albo uruchom zadanie evidence (terminal z katalogu `student/apps/lesson_g_app`):
- `./gradlew :app:bsmEvidence`
1. Jeśli implementacja jest poprawna, w logu zobaczysz krótki kod (5 znaków). To jest odpowiedź do wysłania w komórce G02.


In [ ]:
#@title G02 — Wyślij odpowiedź (auto)
# Wklej 5-znakowy kod z `./gradlew :app:bsmEvidence`.
autosend_text(
    task_id="G02",
    title="G02 — kod zaliczeniowy (5 znaków)",
    placeholder="Wklej tutaj kod, np. K2Q7M",
)


# G03 — Żądanie do backendu bramkowane integralnością (runtime trust)

## Kontekst teoretyczny
Backend nie powinien ufać klientowi tylko dlatego, że „ma właściwy package name”.
W praktyce atakujący może:
- przepakować (repackage) APK,
- zmienić kod, wyłączyć zabezpieczenia, podmienić endpoint,
- zbudować aplikację o tej samej nazwie paczki, ale innym podpisie.

Dlatego w nowoczesnych systemach stosuje się sygnały integralności/proweniencji, a żądania wiąże się z tożsamością aplikacji.
W tym labie nie konfigurujesz prawdziwego Play Integrity API, ale modelujesz podobny mechanizm w sposób testowalny.

## Co masz zrobić (krok po kroku)
1. Otwórz plik: `app/src/main/java/com/example/secretlab/lab/TaskCompletion.kt`.
   Zobacz, jakie 3 warunki są sprawdzane dla `IntegrityState`.
1. Zidentyfikuj miejsce w aplikacji, w którym powstaje żądanie HTTP:
- `app/src/main/java/com/example/secretlab/MainActivity.kt`
  (funkcja `submitAnswer(...)` buduje payload JSON i wysyła POST).
1. Dodaj warstwę „bramki zaufania” przed wysłaniem:
- najpierw oblicz stan integralności/proweniencji aplikacji,
- dopiero potem pozwól na wysyłkę, jeśli warunki są spełnione,
- jeśli nie są spełnione: zastosuj bezpieczny fallback (np. blokada wysyłki + czytelny komunikat dla użytkownika).

## Gdzie w Androidzie sprawdza się tożsamość aplikacji (wskazówki, nie kod rozwiązania)
- `PackageManager` potrafi zwrócić informacje o podpisie aplikacji.
- Dla Android 9+ dostępne jest pole `SigningInfo`.

Linki do dokumentacji:
- PackageManager:
  https://developer.android.com/reference/android/content/pm/PackageManager
- SigningInfo:
  https://developer.android.com/reference/android/content/pm/SigningInfo

## Jak wiązać żądanie z tożsamością aplikacji (binding)
Samo „sprawdzenie czegoś lokalnie” nie wystarcza, jeśli potem wysyłasz żądanie, które da się łatwo podrobić.
W minimalnym modelu wiązania możesz:
- dołączyć do żądania identyfikator oparty o podpis aplikacji (np. skrót certyfikatu),
- dodać element świeżości (nonce / timestamp) i sprawdzać go po stronie serwera (tu: w teście/mocku),
- upewnić się, że bez posiadania tej samej tożsamości aplikacji nie da się wygenerować poprawnego żądania.

## Jak zdobyć kod zaliczeniowy (G03)
1. Uruchom `./gradlew :app:bsmEvidence`.
1. Jeśli implementacja jest poprawna, w logu pojawi się krótki kod (5 znaków). To jest odpowiedź do wysłania w komórce G03.


In [ ]:
#@title G03 — Wyślij odpowiedź (auto)
# Wklej 5-znakowy kod z `./gradlew :app:bsmEvidence`.
autosend_text(
    task_id="G03",
    title="G03 — kod zaliczeniowy (5 znaków)",
    placeholder="Wklej tutaj kod, np. I3B9T",
)


# G04 — Higiena sekretów przy backupie i migracji

## Kontekst teoretyczny
Android oferuje backup i przywracanie danych aplikacji. To bywa wygodne, ale dla sekretów jest krytyczne, bo:
- część sekretów jest *device-bound* (nie wolno migrować),
- tokeny sesyjne i dane uwierzytelniające nie powinny wracać „same z siebie” po reinstalacji,
- dane debugowe i diagnostyczne nie powinny trafiać do backupu.

W tym zadaniu chodzi o świadome ustawienie polityki:
- co wolno backupować,
- czego nie wolno,
- co trzeba odtworzyć bezpiecznie po reinstalacji.

## Co masz zrobić (krok po kroku)
1. Otwórz manifest: `app/src/main/AndroidManifest.xml`.
1. Zwróć uwagę na `android:allowBackup`.
   W starterze jest ustawione na `true`, czyli system może kopiować część danych aplikacji.
1. Wybierz strategię i uzasadnij ją w komentarzu/README (w swoim repo):
- wariant A: wyłączasz backup całkowicie (najprościej, ale tracisz wygodę użytkownika),
- wariant B: zostawiasz backup, ale dodajesz reguły wykluczeń dla sekretów.

## Reguły backupu (co i gdzie)
W zależności od wersji Androida możesz użyć:
- `android:fullBackupContent` (klasyczne reguły Auto Backup)
- `android:dataExtractionRules` (nowsze zasady ekstrakcji danych)

Linki do dokumentacji:
- Auto Backup:
  https://developer.android.com/identity/data/autobackup
- Backup rules:
  https://developer.android.com/identity/data/autobackup#IncludingFiles
- `android:allowBackup`:
  https://developer.android.com/guide/topics/manifest/application-element#allowbackup
- `android:fullBackupContent`:
  https://developer.android.com/guide/topics/manifest/application-element#fullBackupContent
- `android:dataExtractionRules`:
  https://developer.android.com/guide/topics/manifest/application-element#dataExtractionRules

## Skąd wziąć odpowiedź
W kodzie aplikacji jest 5-znakowa wartość sekretu dla labu.
1. Otwórz `app/src/main/java/com/example/secretlab/MainActivity.kt`.
1. Wyszukaj stałe dla zadania 4 (`TASK_4_SECRET_...`).
1. Po wdrożeniu swojej polityki backupu/migracji odczytaj wartość sekretu w sposób przewidziany przez aplikację i wklej ją w komórce G04.

Uwaga: Nie wysyłaj gotowego kodu rozwiązania. Masz pokazać, że rozumiesz *politykę* i potrafisz ją poprawnie ustawić.


In [ ]:
#@title G04 — Wyślij odpowiedź (auto)
# Wklej 5-znakową wartość sekretu dla zadania 4.
autosend_text(
    task_id="G04",
    title="G04 — sekret (5 znaków)",
    placeholder="Wklej tutaj 5-znakowy sekret",
)
